<a href="https://colab.research.google.com/github/rlaqhtjd-2026/group-8-surveying2/blob/main/01_%EA%B3%B5%ED%86%B5%EC%BD%94%EB%93%9C/01_sentinel2_ndci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Google Earth Engine 인증 및 초기화

Google Earth Engine(GEE)을 통해 Sentinel-2 위성 데이터를 활용할 수 있도록 API 인증 및 초기화를 수행합니다.

In [8]:
import ee
import datetime

# Google Earth Engine 인증 및 초기화
try:
    # 사용자의 GCP 프로젝트 ID를 설정하여 초기화를 시도합니다.
    ee.Initialize(project='midyear-precept-508100-s4')
    print("Earth Engine이 성공적으로 초기화되었습니다.")
except Exception as e:
    print("인증을 진행합니다...")
    ee.Authenticate()
    # 인증 후에도 프로젝트 ID를 지정하여 초기화해야 합니다.
    ee.Initialize(project='midyear-precept-508100-s4')
    print("인증 후 Earth Engine이 성공적으로 초기화되었습니다.")

Earth Engine이 성공적으로 초기화되었습니다.


## 2. 위성영상 조회 지역 설정

Sentinel-2 영상을 조회할 연구지역을 `roi`라는 이름으로 설정합니다. ROI는 **관심영역(Region of Interest)**을 뜻합니다.

먼저 Earth Engine 기능을 사용하는 `ee`와 이후 조회 기간 설정에 사용하는 `datetime` 모듈을 불러옵니다.

`ee.Geometry.Rectangle()`은 네 개의 좌표로 사각형 영역을 만듭니다. 좌표 입력 순서는 **[서쪽 경도, 남쪽 위도, 동쪽 경도, 북쪽 위도]**입니다.

이 코드에서는 `[128.95, 35.25, 129.05, 35.40]`을 **물금·매리 일대의 영상 조회를 위한 임시 범위**로 설정합니다. 정확한 채수지점이나 최종 연구구간을 의미하지 않으며, 변경 전·후 채수지점과 종·횡방향 비교 구역을 확인한 뒤 수정해야 합니다.

이 셀은 **GEE 인증·초기화 이후, Sentinel-2 영상 조회 코드 이전에 실행**합니다. 정상적으로 실행되면 ‘영상 조회 지역 설정 완료’라는 문구가 출력됩니다.

In [9]:
# 필요한 모듈
import ee
import datetime

# 위성영상 조회용 임시 범위
# 순서: [서쪽 경도, 남쪽 위도, 동쪽 경도, 북쪽 위도]
roi = ee.Geometry.Rectangle(
    [128.95, 35.25, 129.05, 35.40]
)

print('영상 조회 지역 설정 완료')

영상 조회 지역 설정 완료


## 3. 연구지역 및 분석 기간에 따른 Sentinel-2 영상 조회

현장 클로로필-a 실측자료와 비교하기 위해 연구지역과 겹치는 Sentinel-2 위성영상을 조회합니다. 사용하는 자료는 대기보정된 지표 반사도 영상인 `COPERNICUS/S2_SR_HARMONIZED`입니다.

### 분석 기간 설정

현장자료의 확보 기간과 채수지점 변경을 고려하여 다음 세 기간의 영상을 조회합니다.

- **2021~2024년:** 2021년 1월 1일부터 2024년 12월 31일까지 조회합니다.
- **2025년:** 현장자료 사용 범위에 맞춰 9월 1일부터 12월 31일까지 조회합니다. 1~8월은 결빙에 따른 현장자료 공백과 채수지점 변경 과도기를 고려한 연구 설계에 따라 제외합니다.
- **2026년:** 실행일 현재까지 조회합니다. 이후 현장자료와 연결할 때 실제 확보한 마지막 채수일까지로 제한합니다.

`filterDate()`는 시작일을 포함하고 종료일은 포함하지 않습니다. 따라서 2025년 12월 31일까지 조회하려면 종료일을 `2026-01-01`로 지정합니다.

### 코드의 처리 순서

1. `filterBounds(roi)`로 설정한 연구지역과 겹치는 영상을 선택합니다.
2. `filterDate()`로 세 기간의 영상을 각각 조회합니다.
3. `merge()`로 조회 결과를 하나의 영상 모음으로 합칩니다.
4. `sort('system:time_start')`로 촬영 시각 순서대로 정렬합니다.
5. `size().getInfo()`로 기간별 영상 수와 전체 영상 수를 확인합니다.

### 현재 단계의 결과와 후속 작업

이 코드는 **분석 후보 영상을 조회하는 단계**입니다. 구름·그림자 제거, 수면 구역 선정, NDCI 계산 및 파일 다운로드는 아직 수행하지 않습니다. 조회된 영상 수에는 같은 날짜의 서로 다른 타일이 포함될 수 있으므로 관측 날짜 수와는 구분합니다.

이후 변경 전·후 채수지점 좌표를 각각 적용하여 해당 위치의 NDCI를 추출하고, 현장 채수일과 위성 관측일을 대응시킵니다. 현재 `roi`가 임시 범위라면 두 채수지점과 종·횡방향 비교 구역을 포함하도록 수정해야 합니다.

In [10]:
# 현장자료에 대응할 Sentinel-2 영상 조회
# roi는 우선 기존 임시 범위 사용
# 이후 변경 전·후 채수지점과 비교 구역을 모두 포함하도록 수정

s2_all = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
)

# 2021~2024년
s2_2021_2024 = s2_all.filterDate(
    '2021-01-01', '2025-01-01'
)

# 2025년: 9~12월만 사용
s2_2025 = s2_all.filterDate(
    '2025-09-01', '2026-01-01'
)

# 2026년: 실행일 현재까지 조회
# 나중에 실제 현장자료의 마지막 채수일에 맞춰 제한
end_2026 = min(
    datetime.date.today() + datetime.timedelta(days=1),
    datetime.date(2027, 1, 1)
).isoformat()

s2_2026 = s2_all.filterDate(
    '2026-01-01', end_2026
)

# 세 기간을 하나로 합치기
s2 = (
    s2_2021_2024
    .merge(s2_2025)
    .merge(s2_2026)
    .sort('system:time_start')
)

print('2021~2024년 영상 수:', s2_2021_2024.size().getInfo())
print('2025년 9~12월 영상 수:', s2_2025.size().getInfo())
print('2026년 영상 수:', s2_2026.size().getInfo())
print('전체 영상 수:', s2.size().getInfo())

2021~2024년 영상 수: 584
2025년 9~12월 영상 수: 52
2026년 영상 수: 152
전체 영상 수: 788


## 4. 분석 기간에 해당하는 Sentinel-2 영상 조회

설정한 연구지역 `roi`와 겹치는 Sentinel-2 지표 반사도 영상을 조회합니다.

현장 클로로필-a 자료의 사용 기간에 맞춰 2021~2024년, 2025년 9~12월, 2026년 실행일 현재까지의 영상을 각각 조회한 후 하나로 합칩니다. 2026년 자료는 이후 현장자료와 연결할 때 실제 확보한 마지막 채수일까지로 제한합니다.

조회된 영상 수와 서로 다른 관측 날짜 수를 확인합니다. 같은 날짜에도 여러 타일이 조회될 수 있으므로 두 수는 다를 수 있습니다.

현재는 분석 후보 영상을 조회하는 단계이며, 구름 제거와 NDCI 계산은 다음 단계에서 수행합니다.

In [11]:
# 3. 분석 기간에 해당하는 Sentinel-2 영상 조회

import ee
import datetime

# 앞 단계 실행 여부 확인
if 'roi' not in globals():
    raise ValueError('먼저 연구지역 roi를 설정한 셀을 실행하세요.')

# 연구지역과 겹치는 Sentinel-2 지표 반사도 영상
s2_all = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
)

# 2026년은 실행일 현재까지 조회
end_2026 = min(
    datetime.date.today() + datetime.timedelta(days=1),
    datetime.date(2027, 1, 1)
).isoformat()

# 사용할 기간을 합치기
s2 = (
    s2_all.filterDate('2021-01-01', '2025-01-01')
    .merge(s2_all.filterDate('2025-09-01', '2026-01-01'))
    .merge(s2_all.filterDate('2026-01-01', end_2026))
    .sort('system:time_start')
)

image_count = s2.size().getInfo()

if image_count == 0:
    raise ValueError('조회된 영상이 없습니다. roi와 기간을 확인하세요.')

# 날짜 수 확인: 한국시간 기준
date_list = (
    ee.List(s2.aggregate_array('system:time_start'))
    .map(lambda t: ee.Date(t).format('YYYY-MM-dd', 'Asia/Seoul'))
    .distinct()
    .sort()
    .getInfo()
)

print('조회된 영상 수:', image_count)
print('서로 다른 관측 날짜 수:', len(date_list))
print('처음 10개 관측 날짜:', date_list[:10])

조회된 영상 수: 788
서로 다른 관측 날짜 수: 391
처음 10개 관측 날짜: ['2021-01-04', '2021-01-09', '2021-01-14', '2021-01-19', '2021-01-24', '2021-01-29', '2021-02-03', '2021-02-08', '2021-02-13', '2021-02-18']


## 5. 구름·그림자 제거 및 NDCI 계산

영상에 포함된 SCL(Scene Classification Layer)을 이용해 구름, 구름 그림자, 눈·얼음 및 불량 화소 등을 제외합니다. 이번 예비 처리에서는 SCL 분류값 4·5·6·7에 해당하는 화소를 남깁니다. 이 결과에는 육지도 포함되므로, 수면만을 대상으로 한 결과와 구분합니다.

NDCI 계산에는 적색 밴드 B4와 적색경계 밴드 B5를 사용합니다. B4는 원래 10 m, B5는 20 m 해상도이므로 B4를 평균 집계하여 B5의 20 m 격자에 맞춥니다.

계산식은 NDCI = (B5 − B4) / (B5 + B4)입니다. 분모가 0 이하인 화소는 제외합니다. 계산한 NDCI와 지도 확인용 RGB 밴드를 함께 보관합니다.

이 단계에서 얻는 값은 단위 없는 NDCI이며, 클로로필-a 농도는 아닙니다. 또한 이 전처리는 예비 확인용이며, 최종 분석 전에는 연구지역에서 구름·그림자 제거와 수면 구분이 적절한지 확인해야 합니다.

In [12]:
# 4. 구름·그림자 제거 및 NDCI 계산 함수

def prepare_ndci(image):
    # SCL: 화소의 구름·그림자·수면·육지 등 분류 정보
    scl = image.select('SCL')

    # 예비 품질 마스크
    # 4: 식생, 5: 비식생, 6: 수면, 7: 미분류
    # 수면만 남기는 처리는 지도 확인 단계에서 별도로 수행
    clear_mask = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
        .Or(scl.eq(7))
    )

    # 반사도 변환 및 품질 마스크 적용
    reflectance = (
        image.select(['B2', 'B3', 'B4', 'B5'])
        .multiply(0.0001)
        .updateMask(clear_mask)
    )

    # B5의 원래 20 m 격자
    grid_20m = image.select('B5').projection()

    # B4: 10 m → 평균 집계하여 B5의 20 m 격자에 맞추기
    b4_20m = (
        reflectance.select('B4')
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            maxPixels=64
        )
        .reproject(crs=grid_20m)
    )

    b5_20m = reflectance.select('B5')
    denominator = b5_20m.add(b4_20m)

    # NDCI 계산
    ndci = (
        b5_20m.subtract(b4_20m)
        .divide(denominator)
        .updateMask(denominator.gt(0))
        .rename('NDCI')
    )

    # NDCI, RGB, SCL을 함께 보관
    return (
        ndci
        .addBands(reflectance.select(['B4', 'B3', 'B2']))
        .addBands(scl)
        .copyProperties(
            image,
            ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE']
        )
    )


# 조회한 영상 전체에 함수 적용
s2_ndci = s2.map(prepare_ndci)

print('전처리·NDCI 계산식이 설정되었습니다.')
print('실제 지도나 수치를 요청할 때 GEE 서버에서 계산됩니다.')

전처리·NDCI 계산식이 설정되었습니다.
실제 지도나 수치를 요청할 때 GEE 서버에서 계산됩니다.


## 6. Sentinel-2 영상과 NDCI 지도 확인

조회한 영상 중 전체 타일의 구름량이 가장 적은 영상 한 장을 선택해 처리 결과를 확인합니다. 전체 타일의 구름량이 적더라도 연구지역에 구름이 있을 수 있으므로 RGB 영상과 함께 확인합니다.

지도에는 연구지역 경계, 원본 RGB 영상, 구름·그림자 제거 후 RGB 영상, NDCI 지도를 표시합니다.

수면 NDCI는 예비 확인을 위해 SCL이 수면으로 분류한 화소만 표시합니다. 조류가 많거나 물의 광학 특성이 달라진 구역이 수면으로 분류되지 않을 수 있으므로, 최종 분석에 사용할 수면 범위는 별도로 검토해야 합니다.

NDCI의 색상 범위는 화면 표시용 설정입니다. 색깔 자체가 클로로필-a 농도나 조류경보 단계를 뜻하지 않습니다.

In [13]:
%pip -q install geemap

In [15]:
# 5. 영상 한 장을 선택하여 지도에서 확인

import geemap

# 전체 타일 구름량이 가장 적은 영상 선택
# 연구지역의 구름이 가장 적다는 뜻은 아닙니다.
preview_source = ee.Image(
    s2.sort('CLOUDY_PIXEL_PERCENTAGE').first()
)

preview = ee.Image(prepare_ndci(preview_source))

# 선택 영상 정보
info = ee.Dictionary({
    'image_id': preview_source.get('system:index'),
    'date_kst': preview_source.date().format(
        'YYYY-MM-dd HH:mm', 'Asia/Seoul'
    ),
    'cloud_percent': preview_source.get('CLOUDY_PIXEL_PERCENTAGE')
}).getInfo()

print('선택 영상 ID:', info['image_id'])
print('촬영 시각(한국시간):', info['date_kst'])
print('전체 타일 구름량(%):', info['cloud_percent'])

# RGB 표시 설정: 반사도 기준
rgb_style = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.2
}

# NDCI 표시 설정: 농도나 경보 기준이 아님
ndci_style = {
    'min': -0.3,
    'max': 0.5,
    'palette': ['2166ac', 'ffffbf', '1a9850']
}

# 예비 수면 마스크: SCL 6 = 수면
water_ndci = (
    preview.select('NDCI')
    .updateMask(preview.select('SCL').eq(6))
)

# 지도 생성
m = geemap.Map()
m.centerObject(roi, 12)

# 원본 RGB: 구름 제거 결과와 비교하기 위한 영상
original_rgb = (
    preview_source.select(['B4', 'B3', 'B2'])
    .multiply(0.0001)
)

m.addLayer(
    original_rgb.clip(roi),
    rgb_style,
    '1. 원본 RGB',
    True
)

m.addLayer(
    preview.select(['B4', 'B3', 'B2']).clip(roi),
    rgb_style,
    '2. 품질 마스크 적용 RGB',
    False
)

m.addLayer(
    preview.select('NDCI').clip(roi),
    ndci_style,
    '3. NDCI - 육지 포함',
    False
)

m.addLayer(
    water_ndci.clip(roi),
    ndci_style,
    '4. NDCI - SCL 수면만',
    False
)

# 연구지역 경계선
roi_outline = ee.FeatureCollection(
    [ee.Feature(roi)]
).style(
    color='FF0000',
    fillColor='00000000',
    width=2
)

m.addLayer(roi_outline, {}, '연구지역 경계')

display(m)

선택 영상 ID: 1_1_20240518T020649_20240518T020649_T52SDE
촬영 시각(한국시간): 2024-05-18 11:17
전체 타일 구름량(%): 0.000511


Map(center=[35.324987100105936, 128.99999999999704], controls=(WidgetControl(options=['position', 'transparent…